# Phân Tích Cảm Xúc Đa Khía Cạnh (ABSA): Huấn Luyện PhoBERT
Chi tiết các bước Fine-tune mô hình `vinai/phobert-base-v2` cho bài toán Aspect-Based Sentiment Analysis. Mô hình sẽ đồng thời phân loại Cảm xúc chung (Sentiment) và 6 Khía cạnh đánh giá (Aspects).
Sử dụng Focal Loss xử lý mất cân bằng phân lớp và t-SNE để trực quan hóa không gian Embeddings.

## 1. Import Thư Viện Chuyên Dụng

In [7]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.manifold import TSNE

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from collections import Counter
import warnings

warnings.filterwarnings('ignore')


## 2. Thiết lập Môi trường & Tham số

In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Thiết bị đang sử dụng:', device)


Thiết bị đang sử dụng: cuda


In [10]:
# Tham số cố định của mô hình
MODEL_NAME = 'vinai/phobert-base-v2'
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 4
LEARNING_RATE = 2e-5


In [11]:
# Định nghĩa các tập nhãn
SENTIMENT_LABELS = {0: 'Tiêu cực', 1: 'Trung lập', 2: 'Tích cực'}
ASPECT_LABELS = {0: 'Tiêu cực', 1: 'Trung lập', 2: 'Tích cực', 3: 'Không nhắc đến'}
ASPECT_COLS = ['as_content', 'as_physical', 'as_price', 'as_packaging', 'as_delivery', 'as_service']
TARGET_COLS = ['sentiment_llm'] + ASPECT_COLS


## 3. Nạp và Chuẩn bị Dữ liệu

In [12]:
# Nạp dữ liệu từ 2 file JSON riêng biệt
TRAIN_PATH = '/kaggle/input/datasets/phtnguyn1ytj/tiki-book-reviews/train_clean.json'
TEST_PATH  = '/kaggle/input/datasets/phtnguyn1ytj/tiki-book-reviews/test_clean.json'

train_raw = pd.read_json(TRAIN_PATH)
test_raw  = pd.read_json(TEST_PATH)

print('train_raw shape:', train_raw.shape)
print('test_raw  shape:', test_raw.shape)
print('Columns:', train_raw.columns.tolist())


train_raw shape: (10697, 16)
test_raw  shape: (2676, 16)
Columns: ['review_id', 'rating', 'review_title', 'content', 'product_id', 'product_name', 'category', 'created_at', 'sentiment_llm', 'as_content', 'as_physical', 'as_price', 'as_packaging', 'as_delivery', 'as_service', 'content_raw']


In [14]:
# Chuẩn hoá cột văn bản (hỗ trợ cả 'text' lẫn 'content_clean')
def prepare_df(df):
    text_col = 'text' if 'text' in df.columns else 'content'
    df = df[[text_col] + TARGET_COLS].dropna(subset=[text_col, 'sentiment_llm']).copy()
    df = df.rename(columns={text_col: 'text'})

    # Ép kiểu Sentiment
    df['sentiment_llm'] = df['sentiment_llm'].astype(int)
    df = df[df['sentiment_llm'].isin([0, 1, 2])]

    # Xử lý 6 cột Aspect: NaN -> 3 (Không nhắc đến)
    for col in ASPECT_COLS:
        df[col] = df[col].fillna(3).astype(int)
        df[col] = df[col].apply(lambda x: x if x in [0, 1, 2, 3] else 3)

    return df.reset_index(drop=True)

full_train_df = prepare_df(train_raw)
test_df       = prepare_df(test_raw)

print('full_train_df:', len(full_train_df), '| test_df:', len(test_df))


full_train_df: 10697 | test_df: 2676


In [15]:
display(full_train_df.head(3))
display(test_df.head(3))


,text,sentiment_llm,as_content,as_physical,as_price,as_packaging,as_delivery,as_service
0,"Seri truyện của tác giả này bé nhà mình có đủ,...",2,3,2,3,3,3,3
1,Hình như dịch vụ vận chuyển của Tiki ngày càng...,0,3,0,3,3,0,3
2,"Sản phẩm hơi cũ và ố ơ cạnh trang sách, nhưng ...",2,2,3,1,3,3,3


,text,sentiment_llm,as_content,as_physical,as_price,as_packaging,as_delivery,as_service
0,bổ ích cho cuộc sống hiện đại,2,2,3,3,3,3,3
1,"Mang tên sách "" thực hành tư duy thiết kế"" như...",0,0,3,3,3,3,3
2,Hay. Đọc mà khóc quá trời!,2,3,3,3,3,3,3


## 4. Phân tách tập Train / Val (từ train_clean.json)

In [16]:
# Chia 85% Train / 15% Val từ train_clean.json
# test_clean.json giữ nguyên làm tập Test độc lập
train_df, val_df = train_test_split(
    full_train_df,
    test_size=0.15,
    random_state=42,
    stratify=full_train_df['sentiment_llm']
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')


Train: 9,092 | Val: 1,605 | Test: 2,676


## 5. Áp dụng Tokenizer và Đóng gói Tensor 7 Nhãn

In [17]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [19]:
def tokenize_function(examples):
    tokenized = tokenizer(examples['text'], padding='max_length', truncation=True, max_length=MAX_LENGTH)

    labels_batch = []
    for i in range(len(examples['text'])):
        sample_labels = [examples[col][i] for col in TARGET_COLS]
        labels_batch.append(sample_labels)

    tokenized['labels'] = labels_batch
    return tokenized


In [20]:
# Chuyển Pandas -> HuggingFace Dataset
COLS_TO_REMOVE = ['text'] + TARGET_COLS

train_dataset = Dataset.from_pandas(train_df)
val_dataset   = Dataset.from_pandas(val_df)
test_dataset  = Dataset.from_pandas(test_df)

# Batched Tokenization
train_encoded = train_dataset.map(tokenize_function, batched=True, remove_columns=COLS_TO_REMOVE)
val_encoded   = val_dataset.map(tokenize_function,   batched=True, remove_columns=COLS_TO_REMOVE)
test_encoded  = test_dataset.map(tokenize_function,  batched=True, remove_columns=COLS_TO_REMOVE)

# Ép Tensor PyTorch
for ds in [train_encoded, val_encoded, test_encoded]:
    ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])


Map:   0%|          | 0/9092 [00:00<?, ? examples/s]

Map:   0%|          | 0/1605 [00:00<?, ? examples/s]

Map:   0%|          | 0/2676 [00:00<?, ? examples/s]

## 6. Khởi tạo Mô hình PhoBERT (27 Output Units)

In [21]:
# Kiến trúc: 27 output units
# - [0:3]  : sentiment_llm (3 class)
# - [3:27] : 6 aspect x 4 class = 24 logits
NUM_LABELS = 3 + (6 * 4)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)
model.to(device)
print('num_labels:', model.config.num_labels)


pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

num_labels: 27


## 7. Multi-Task Focal Loss

In [22]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()


In [23]:
class ABSAMultiTaskTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')          # [batch, 7]
        outputs = model(**inputs)
        logits = outputs.logits                # [batch, 27]

        loss_fct = FocalLoss()

        # 1. Sentiment loss
        loss_sentiment = loss_fct(logits[:, :3], labels[:, 0])

        # 2. Aspect loss (6 tasks, averaged)
        asp_logits = logits[:, 3:].view(-1, 6, 4)  # [batch, 6, 4]
        loss_aspects = sum(
            loss_fct(asp_logits[:, i, :], labels[:, i + 1])
            for i in range(6)
        ) / 6.0

        total_loss = loss_sentiment + loss_aspects
        return (total_loss, outputs) if return_outputs else total_loss


In [24]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    pred_sentiment = np.argmax(logits[:, :3], axis=-1)
    pred_aspects   = np.argmax(logits[:, 3:].reshape(-1, 6, 4), axis=-1)

    true_sentiment = labels[:, 0]
    true_aspects   = labels[:, 1:]

    all_true = np.concatenate([true_sentiment.flatten(), true_aspects.flatten()])
    all_pred = np.concatenate([pred_sentiment.flatten(), pred_aspects.flatten()])

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_true, all_pred, average='macro', zero_division=0
    )
    acc = accuracy_score(all_true, all_pred)

    return {'accuracy': acc, 'f1_macro': f1, 'precision': precision, 'recall': recall}


## 8. Huấn Luyện

In [25]:
training_args = TrainingArguments(
    output_dir='./absa_results',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    warmup_steps=200,
    weight_decay=0.01,
    logging_dir='./absa_logs',
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    report_to='none'
)

trainer = ABSAMultiTaskTrainer(
    model=model,
    args=training_args,
    train_dataset=train_encoded,
    eval_dataset=val_encoded,
    compute_metrics=compute_metrics
)

print('Bắt đầu huấn luyện ABSA Model...')
trainer.train()
print('Huấn luyện hoàn tất.')


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Bắt đầu huấn luyện ABSA Model...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision,Recall
1,0.395969,0.339945,0.844059,0.675329,0.768849,0.622000
2,0.275771,0.284450,0.877793,0.737557,0.792311,0.701990
3,0.225106,0.269717,0.890254,0.754244,0.793779,0.730472
4,0.179590,0.262738,0.899065,0.768826,0.812085,0.743151


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Huấn luyện hoàn tất.


## 9. Đánh giá trên tập Test (test_clean.json)

In [26]:
# Kết quả trên Val (dùng để chọn checkpoint)
val_result = trainer.evaluate(val_encoded)
print('Val metrics:', val_result)


Val metrics: {'eval_loss': 0.26273810863494873, 'eval_accuracy': 0.8990654205607477, 'eval_f1_macro': 0.7688260833986957, 'eval_precision': 0.8120854209776492, 'eval_recall': 0.7431508935513729, 'eval_runtime': 5.5122, 'eval_samples_per_second': 291.174, 'eval_steps_per_second': 9.252, 'epoch': 4.0}


In [27]:
# Kết quả trên Test độc lập
test_result = trainer.evaluate(test_encoded)
print('Test metrics:', test_result)


Test metrics: {'eval_loss': 0.2662413418292999, 'eval_accuracy': 0.8943519111680547, 'eval_f1_macro': 0.7609190673022043, 'eval_precision': 0.8083912214121604, 'eval_recall': 0.7346409089654692, 'eval_runtime': 9.145, 'eval_samples_per_second': 292.617, 'eval_steps_per_second': 9.185, 'epoch': 4.0}


In [28]:
raw_pred, _, _ = trainer.predict(test_encoded)

pred_sent = np.argmax(raw_pred[:, :3], axis=1)
pred_asps = np.argmax(raw_pred[:, 3:].reshape(-1, 6, 4), axis=2)

true_sent = test_df['sentiment_llm'].values
true_asps = test_df[ASPECT_COLS].values


In [29]:
print('=== BÁO CÁO CẢM XÚC CHUNG (TEST SET) ===')
print(classification_report(true_sent, pred_sent,
      target_names=['Tiêu cực', 'Trung lập', 'Tích cực'], zero_division=0))


=== BÁO CÁO CẢM XÚC CHUNG (TEST SET) ===
              precision    recall  f1-score   support

    Tiêu cực       0.94      0.95      0.95      1402
   Trung lập       0.69      0.67      0.68       437
    Tích cực       0.91      0.92      0.92       837

    accuracy                           0.89      2676
   macro avg       0.85      0.85      0.85      2676
weighted avg       0.89      0.89      0.89      2676



In [30]:
print('\n=== BÁO CÁO 6 KHÍA CẠNH (TEST SET) ===')
print(classification_report(true_asps.flatten(), pred_asps.flatten(),
      target_names=['Tiêu cực', 'Trung lập', 'Tích cực', 'Không nhắc đến'], zero_division=0))



=== BÁO CÁO 6 KHÍA CẠNH (TEST SET) ===
                precision    recall  f1-score   support

      Tiêu cực       0.67      0.64      0.65      1053
     Trung lập       0.61      0.16      0.25       601
      Tích cực       0.74      0.70      0.72      1443
Không nhắc đến       0.93      0.97      0.95     12959

      accuracy                           0.89     16056
     macro avg       0.74      0.62      0.64     16056
  weighted avg       0.88      0.89      0.88     16056



## 10. Trực quan hóa Embeddings (t-SNE)

In [31]:
word_embeddings = model.roberta.embeddings.word_embeddings.weight.detach().cpu().numpy()


In [32]:
all_words = ' '.join(train_df['text'].tolist()).split()
top_words_freq = Counter(all_words).most_common(700)


In [33]:
words_to_plot, vectors_to_plot, frequencies, sentiments = [], [], [], []

for w, freq in top_words_freq:
    token_id = tokenizer.convert_tokens_to_ids(tokenizer.tokenize(w))
    if len(token_id) == 1 and token_id[0] != tokenizer.unk_token_id:
        words_to_plot.append(w)
        vectors_to_plot.append(word_embeddings[token_id[0]])
        frequencies.append(freq)
        sub_df = train_df[train_df['text'].str.contains(w, na=False, regex=False)]
        dominant_sent = sub_df['sentiment_llm'].mode()[0] if len(sub_df) > 0 else 1
        sentiments.append(SENTIMENT_LABELS[dominant_sent])


In [34]:
vectors_to_plot = np.array(vectors_to_plot)
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
tsne_result = tsne.fit_transform(vectors_to_plot)


In [36]:
df_tsne = pd.DataFrame({
    'word': words_to_plot,
    'x': tsne_result[:, 0],
    'y': tsne_result[:, 1],
    'frequency': frequencies,
    'sentiment': sentiments
})


In [38]:
fig = px.scatter(
    df_tsne, x='x', y='y',
    color='sentiment',
    color_discrete_map={'Tiêu cực': '#EF553B', 'Trung lập': '#636EFA', 'Tích cực': '#00CC96'},
    size=[max(10, np.sqrt(f)) for f in df_tsne['frequency']],
    hover_name='word',
    hover_data={'x': False, 'y': False, 'frequency': True, 'sentiment': True},
    title='Biểu diễn Phân cụm Ngữ Nghĩa (PhoBERT Embeddings t-SNE)',
    opacity=0.8
)
fig.update_layout(width=900, height=700, plot_bgcolor='white')
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig.show()


## 11. API Dự đoán Đầu Cuối (Inference Pipeline)

In [39]:
def predict_absa(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True,
                       padding=True, max_length=128).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits

    sent_logits = logits[:, :3]
    asp_logits  = logits[:, 3:].reshape(1, 6, 4)

    sent_pred = torch.argmax(sent_logits, dim=-1).item()
    sent_conf = torch.max(F.softmax(sent_logits, dim=-1)).item()

    asp_preds = {}
    for i, col in enumerate(ASPECT_COLS):
        col_name = col.replace('as_', '').title()
        p_idx = torch.argmax(asp_logits[0, i, :]).item()
        p_conf = torch.max(F.softmax(asp_logits[0, i, :], dim=-1)).item()
        if p_idx != 3:
            asp_preds[col_name] = f'{ASPECT_LABELS[p_idx]} ({p_conf:.0%})'

    return SENTIMENT_LABELS[sent_pred], sent_conf, asp_preds


In [40]:
samples = [
    'Sách rất hay, nội dung sâu sắc nhưng sách nhận được bị móp méo kinh khủng, shipper lề mề.',
    'Giá hơi đắt so với thị trường nhưng đóng gói đẹp, đọc khá ổn',
    'Gói hàng bình thường, nội dung chưa đọc không biết thế nào.',
    'Truyện đỉnh chóp, giao trong 2 giờ.'
]

print('KẾT QUẢ INFERENCE ĐA NHIỆM:\n' + '='*50)
for s in samples:
    sent_label, sent_conf, aspects = predict_absa(s)
    print(f"Bình luận: '{s}'")
    print(f'>> Cảm xúc Chung: {sent_label} (Tin cậy: {sent_conf:.0%})')
    if aspects:
        print('>> Chi tiết Khía cạnh:')
        for k, v in aspects.items():
            print(f'   - {k.ljust(10)}: {v}')
    else:
        print('>> Chi tiết: Không nhắc đến khía cạnh cụ thể nào.')
    print('-' * 50)


KẾT QUẢ INFERENCE ĐA NHIỆM:
Bình luận: 'Sách rất hay, nội dung sâu sắc nhưng sách nhận được bị móp méo kinh khủng, shipper lề mề.'
>> Cảm xúc Chung: Tiêu cực (Tin cậy: 68%)
>> Chi tiết Khía cạnh:
   - Physical  : Tiêu cực (62%)
--------------------------------------------------
Bình luận: 'Giá hơi đắt so với thị trường nhưng đóng gói đẹp, đọc khá ổn'
>> Cảm xúc Chung: Trung lập (Tin cậy: 83%)
>> Chi tiết Khía cạnh:
   - Packaging : Tích cực (53%)
--------------------------------------------------
Bình luận: 'Gói hàng bình thường, nội dung chưa đọc không biết thế nào.'
>> Cảm xúc Chung: Trung lập (Tin cậy: 88%)
>> Chi tiết: Không nhắc đến khía cạnh cụ thể nào.
--------------------------------------------------
Bình luận: 'Truyện đỉnh chóp, giao trong 2 giờ.'
>> Cảm xúc Chung: Tích cực (Tin cậy: 92%)
>> Chi tiết Khía cạnh:
   - Content   : Tích cực (52%)
--------------------------------------------------


## 12. Lưu Trọng Số Tốt Nhất

In [41]:
final_dir = './absa_results/final_model'
model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)
print('Saved final model successfully!')


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved final model successfully!
